# Pipeline

In [9]:
from pathlib import Path

from lana_nlp.preprocessing.data_loader import LyricsDataLoader
from lana_nlp.preprocessing.text_cleaner import TextCleaner
from lana_nlp.features.lyrics_features import LyricsFeatures
from lana_nlp.analysis.lyrics_analyzer import LyricsAnalyzer
from lana_nlp.analysis.readability import ReadabilityAnalyzer
from lana_nlp.analysis.sentiment import SentimentAnalyzer
from lana_nlp.analysis.statistics import StatisticsAnalyzer
from lana_nlp.analysis.vocabulary import VocabularyAnalyzer
from numpy.ma.extras import column_stack

## Load the lyrics

In [10]:
loader = LyricsDataLoader(
    Path("../data/raw/lyrics.csv")
)

df = loader.load()

print(df.shape)
df.head()

(154, 5)


,artist,album,song,year,lyrics
0,Lana Del Rey,Sirens,Drive By,2006,There was a drive-by Sunday night\nMost of us ...
1,Lana Del Rey,Sirens,River Road,2006,"Another day is over, another day is done\nAnd ..."
2,Lana Del Rey,Sirens,A Star for Nick,2006,"Well, you know it and I know it, I'm gonna be ..."
3,Lana Del Rey,Sirens,My Momma,2006,My momma wouldn't say you were a nice guy\nBut...
4,Lana Del Rey,Sirens,Bad Disease,2006,"Well, there's somethin' about watchin' a crime..."


## Clean the lyrics (basic)

In [11]:
cleaner = TextCleaner()

df["basic_cleaned_lyrics"] = df["lyrics"].apply(cleaner.basic_clean)
df.head()

,artist,album,song,year,lyrics,basic_cleaned_lyrics
0,Lana Del Rey,Sirens,Drive By,2006,There was a drive-by Sunday night\nMost of us ...,there was a driveby sunday night most of us we...
1,Lana Del Rey,Sirens,River Road,2006,"Another day is over, another day is done\nAnd ...",another day is over another day is done and no...
2,Lana Del Rey,Sirens,A Star for Nick,2006,"Well, you know it and I know it, I'm gonna be ...",well you know it and i know it im gonna be a s...
3,Lana Del Rey,Sirens,My Momma,2006,My momma wouldn't say you were a nice guy\nBut...,my momma wouldnt say you were a nice guy but y...
4,Lana Del Rey,Sirens,Bad Disease,2006,"Well, there's somethin' about watchin' a crime...",well theres somethin about watchin a crime tha...


## NLP cleaned lyrics

In [12]:
df["nlp_cleaned_lyrics"] = df["basic_cleaned_lyrics"].apply(cleaner.nlp_clean)
df.head()

,artist,album,song,year,lyrics,basic_cleaned_lyrics,nlp_cleaned_lyrics
0,Lana Del Rey,Sirens,Drive By,2006,There was a drive-by Sunday night\nMost of us ...,there was a driveby sunday night most of us we...,"[driveby, sunday, night, u, bed, right, turned..."
1,Lana Del Rey,Sirens,River Road,2006,"Another day is over, another day is done\nAnd ...",another day is over another day is done and no...,"[another, day, another, day, done, im, gettin,..."
2,Lana Del Rey,Sirens,A Star for Nick,2006,"Well, you know it and I know it, I'm gonna be ...",well you know it and i know it im gonna be a s...,"[well, know, know, im, gon, na, star, wont, wo..."
3,Lana Del Rey,Sirens,My Momma,2006,My momma wouldn't say you were a nice guy\nBut...,my momma wouldnt say you were a nice guy but y...,"[momma, wouldnt, say, nice, guy, youre, 40, jo..."
4,Lana Del Rey,Sirens,Bad Disease,2006,"Well, there's somethin' about watchin' a crime...",well theres somethin about watchin a crime tha...,"[well, there, somethin, watchin, crime, make, ..."


In [13]:
df.columns

Index(['artist', 'album', 'song', 'year', 'lyrics', 'basic_cleaned_lyrics',
       'nlp_cleaned_lyrics'],
      dtype='str')

## Create Analyzers

In [14]:
analyzer = LyricsAnalyzer(
    df,
    basic_text_column="basic_cleaned_lyrics",
    nlp_text_column="nlp_cleaned_lyrics",
)

df = analyzer.analyze()

In [15]:
df.shape

(154, 12)

In [16]:
df.columns.tolist()

['artist',
 'album',
 'song',
 'year',
 'lyrics',
 'basic_cleaned_lyrics',
 'nlp_cleaned_lyrics',
 'word_count',
 'unique_words',
 'syllable_count',
 'line_count',
 'reading_minutes']

In [17]:
df.head()

,artist,album,song,year,lyrics,basic_cleaned_lyrics,nlp_cleaned_lyrics,word_count,unique_words,syllable_count,line_count,reading_minutes
0,Lana Del Rey,Sirens,Drive By,2006,There was a drive-by Sunday night\nMost of us ...,there was a driveby sunday night most of us we...,"[driveby, sunday, night, u, bed, right, turned...",248,97,295,1,1.240
1,Lana Del Rey,Sirens,River Road,2006,"Another day is over, another day is done\nAnd ...",another day is over another day is done and no...,"[another, day, another, day, done, im, gettin,...",166,64,211,1,0.830
2,Lana Del Rey,Sirens,A Star for Nick,2006,"Well, you know it and I know it, I'm gonna be ...",well you know it and i know it im gonna be a s...,"[well, know, know, im, gon, na, star, wont, wo...",84,44,98,1,0.420
3,Lana Del Rey,Sirens,My Momma,2006,My momma wouldn't say you were a nice guy\nBut...,my momma wouldnt say you were a nice guy but y...,"[momma, wouldnt, say, nice, guy, youre, 40, jo...",277,114,322,1,1.385
4,Lana Del Rey,Sirens,Bad Disease,2006,"Well, there's somethin' about watchin' a crime...",well theres somethin about watchin a crime tha...,"[well, there, somethin, watchin, crime, make, ...",227,109,268,1,1.135


In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 154 entries, 0 to 153
Data columns (total 12 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   artist                154 non-null    str    
 1   album                 154 non-null    str    
 2   song                  154 non-null    str    
 3   year                  154 non-null    int64  
 4   lyrics                153 non-null    str    
 5   basic_cleaned_lyrics  154 non-null    str    
 6   nlp_cleaned_lyrics    154 non-null    object 
 7   word_count            154 non-null    int64  
 8   unique_words          154 non-null    int64  
 9   syllable_count        154 non-null    int64  
 10  line_count            154 non-null    int64  
 11  reading_minutes       154 non-null    float64
dtypes: float64(1), int64(5), object(1), str(5)
memory usage: 14.6+ KB
